# Comprehensive EDA Pipeline
## Automated Data Profiling & Report Generation

**Supported Formats:** CSV, Excel, JSON, Parquet, Feather, Pickle, TSV

This notebook automatically generates:
- YData Profiling HTML Report
- Sweetviz HTML Report
- AutoViz Visualizations
- D-Tale Interactive Dashboard (in-notebook)

---

## 1. Import Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime
import chardet
import traceback

warnings.filterwarnings('ignore')

# CONFIGURATION - MODIFY THIS
DATA_FILE = 'datasets/wine+quality/winequality-red-edited.csv'  # ← CHANGE THIS TO YOUR FILE
OUTPUT_DIR = 'reports'
CORRELATION_THRESHOLD = 0.5

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✓ Libraries loaded')

✓ Libraries loaded


## 2. Helper Functions

In [2]:
def detect_encoding(file_path):
    try:
        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read(10000))
        return result['encoding'] or 'utf-8'
    except:
        return 'utf-8'

def detect_delimiter(file_path, encoding):
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            sample = f.read(10000)
        delimiters = [',', ';', '\t', '|']
        return max(delimiters, key=sample.count)
    except:
        return ','

def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found: {file_path}')
    
    ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if ext in ['.csv', '.tsv']:
            enc = detect_encoding(file_path)
            delim = detect_delimiter(file_path, enc)
            df = pd.read_csv(file_path, encoding=enc, sep=delim)
        elif ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        elif ext == '.json':
            df = pd.read_json(file_path)
        elif ext == '.parquet':
            df = pd.read_parquet(file_path)
        elif ext == '.feather':
            df = pd.read_feather(file_path)
        elif ext in ['.pkl', '.pickle']:
            df = pd.read_pickle(file_path)
        else:
            raise ValueError(f'Unsupported format: {ext}')
        return df
    except Exception as e:
        print(f'Error loading data: {e}')
        raise

print('✓ Helper functions ready')

✓ Helper functions ready


## 3. Load Dataset

In [3]:
start_time = datetime.now()

try:
    df = load_data(DATA_FILE)
    dataset_name = os.path.splitext(os.path.basename(DATA_FILE))[0]
    dataset_output_dir = os.path.join(OUTPUT_DIR, dataset_name)
    Path(dataset_output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f'✓ Data loaded successfully!')
    print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
    print(f'  Output: {dataset_output_dir}')
except Exception as e:
    print(f'✗ Failed to load: {e}')
    sys.exit(1)

✓ Data loaded successfully!
  Shape: 3,961 rows × 12 columns
  Memory: 0.36 MB
  Output: reports/winequality-white-edited


## 4. Dataset Overview

In [4]:
print('\n' + '='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nColumn Information:')
print(df.dtypes)

print('\nFirst 5 rows:')
df.head()


DATASET OVERVIEW

Shape: 3,961 rows × 12 columns
Memory Usage: 0.36 MB

Column Information:
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

First 5 rows:


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,4
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,4
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,4
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,4
4,6.2,0.32,0.16,7.0,0.045,30.0,136.0,0.9949,3.18,0.47,9.6,4


## 5. Data Types & Missing Values

In [5]:
print('\n' + '='*80)
print('DATA TYPES & MISSING VALUES')
print('='*80)

info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Null': df.isnull().sum().values,
    'Null %': (df.isnull().sum().values / len(df) * 100).round(2)
})

print('\n', info_df.to_string(index=False))
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')


DATA TYPES & MISSING VALUES

               Column    Type  Non-Null  Null  Null %
       fixed_acidity float64      3961     0     0.0
    volatile_acidity float64      3961     0     0.0
         citric_acid float64      3961     0     0.0
      residual_sugar float64      3961     0     0.0
           chlorides float64      3961     0     0.0
 free_sulfur_dioxide float64      3961     0     0.0
total_sulfur_dioxide float64      3961     0     0.0
             density float64      3961     0     0.0
                  pH float64      3961     0     0.0
           sulphates float64      3961     0     0.0
             alcohol float64      3961     0     0.0
             quality   int64      3961     0     0.0

Total Missing Values: 0


## 6. Numerical Statistics

In [6]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    print('\n' + '='*80)
    print('NUMERICAL STATISTICS')
    print('='*80)
    stats = df[numerical_cols].describe(include='all').T
    stats['skew'] = df[numerical_cols].skew()
    stats['kurtosis'] = df[numerical_cols].kurtosis()
    print('\n', stats.round(4))
else:
    print('\nNo numerical columns found.')


NUMERICAL STATISTICS

                        count      mean      std     min       25%       50%  \
fixed_acidity         3961.0    6.8393   0.8669  3.8000    6.3000    6.8000   
volatile_acidity      3961.0    0.2805   0.1034  0.0800    0.2100    0.2600   
citric_acid           3961.0    0.3343   0.1224  0.0000    0.2700    0.3200   
residual_sugar        3961.0    5.9148   4.8616  0.6000    1.6000    4.7000   
chlorides             3961.0    0.0459   0.0231  0.0090    0.0350    0.0420   
free_sulfur_dioxide   3961.0   34.8892  17.2100  2.0000   23.0000   33.0000   
total_sulfur_dioxide  3961.0  137.1935  43.1291  9.0000  106.0000  133.0000   
density               3961.0    0.9938   0.0029  0.9871    0.9916    0.9935   
pH                    3961.0    3.1955   0.1515  2.7200    3.0900    3.1800   
sulphates             3961.0    0.4904   0.1135  0.2200    0.4100    0.4800   
alcohol               3961.0   10.5894   1.2171  8.0000    9.5000   10.4000   
quality               3961.0

## 7. Categorical Statistics

In [7]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    print('\n' + '='*80)
    print('CATEGORICAL STATISTICS')
    print('='*80)
    for col in categorical_cols:
        print(f'\n{col}:')
        print(f'  Unique: {df[col].nunique()}')
        print(f'  Top 5 values:')
        print(df[col].value_counts().head().to_string())
else:
    print('\nNo categorical columns found.')


No categorical columns found.


## 8. Correlation Matrix

In [8]:
if len(numerical_cols) > 1:
    print('\n' + '='*80)
    print('CORRELATION MATRIX')
    print('='*80)
    corr_matrix = df[numerical_cols].corr()
    print('\nFull Correlation Matrix:')
    print(corr_matrix.round(3))
    
    print(f'\nHigh Correlations (abs > {CORRELATION_THRESHOLD}):')
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > CORRELATION_THRESHOLD:
                high_corr.append({
                    'Column 1': corr_matrix.columns[i],
                    'Column 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr:
        print(pd.DataFrame(high_corr).to_string(index=False))
    else:
        print(f'No correlations above {CORRELATION_THRESHOLD} threshold.')
else:
    print('\nNot enough numerical columns for correlation analysis.')


CORRELATION MATRIX

Full Correlation Matrix:
                      fixed_acidity  volatile_acidity  citric_acid  \
fixed_acidity                 1.000            -0.019        0.299   
volatile_acidity             -0.019             1.000       -0.163   
citric_acid                   0.299            -0.163        1.000   
residual_sugar                0.084             0.098        0.106   
chlorides                     0.024             0.086        0.133   
free_sulfur_dioxide          -0.058            -0.102        0.092   
total_sulfur_dioxide          0.082             0.102        0.123   
density                       0.266             0.061        0.160   
pH                           -0.431            -0.047       -0.183   
sulphates                    -0.017            -0.021        0.049   
alcohol                      -0.111             0.047       -0.077   
quality                      -0.125            -0.191        0.007   

                      residual_sugar  chlor

## 9. Generate panda profiling

In [9]:
print('\n' + '='*80)
print('DATA PROFILING REPORT')
print('='*80)

try:
    print('\nGenerating data profile summary...')
    
    profile_file = os.path.join(dataset_output_dir, 'data_profile_report.html')
    
    html_content = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Data Profile Report - {dataset_name}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
            .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; }}
            h1 {{ color: #333; border-bottom: 3px solid #007bff; padding-bottom: 10px; }}
            h2 {{ color: #555; margin-top: 30px; }}
            table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
            th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
            th {{ background-color: #007bff; color: white; }}
            tr:hover {{ background-color: #f9f9f9; }}
            .stat-box {{ display: inline-block; margin: 10px; padding: 15px; background: #f9f9f9; border-left: 4px solid #007bff; }}
            .section {{ margin: 20px 0; padding: 15px; background: #f9f9f9; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Data Profile Report: {dataset_name}</h1>
            
            <div class="section">
                <h2>📈 Dataset Overview</h2>
                <div class="stat-box">
                    <strong>Rows:</strong> {df.shape[0]:,}
                </div>
                <div class="stat-box">
                    <strong>Columns:</strong> {df.shape[1]}
                </div>
                <div class="stat-box">
                    <strong>Memory:</strong> {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB
                </div>
                <div class="stat-box">
                    <strong>Missing Values:</strong> {df.isnull().sum().sum()}
                </div>
            </div>
            
            <h2>📋 Column Information</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Type</th>
                    <th>Non-Null</th>
                    <th>Null</th>
                    <th>Null %</th>
                </tr>
    '''
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].dtype}</td>
                    <td>{df[col].count():,}</td>
                    <td>{null_count}</td>
                    <td>{null_pct:.2f}%</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>🔢 Numerical Statistics</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Mean</th>
                    <th>Std</th>
                    <th>Min</th>
                    <th>Max</th>
                </tr>
    '''
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numerical_cols:
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].mean():.4f}</td>
                    <td>{df[col].std():.4f}</td>
                    <td>{df[col].min():.4f}</td>
                    <td>{df[col].max():.4f}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>📊 Top Values (Categorical)</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Unique</th>
                    <th>Top Value</th>
                    <th>Frequency</th>
                </tr>
    '''
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        top_val = df[col].value_counts().index[0] if len(df[col].value_counts()) > 0 else 'N/A'
        top_count = df[col].value_counts().values[0] if len(df[col].value_counts()) > 0 else 0
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].nunique()}</td>
                    <td>{top_val}</td>
                    <td>{top_count}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <footer style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; text-align: center; color: #666;">
                <p>Generated automatically using Python Pandas</p>
            </footer>
        </div>
    </body>
    </html>
    '''
    
    with open(profile_file, 'w') as f:
        f.write(html_content)
    
    print(f'✓ Custom profiling report saved: {profile_file}')
    if os.path.exists(profile_file):
        file_size_mb = os.path.getsize(profile_file) / 1024**2
        print(f'  File size: {file_size_mb:.2f} MB')
        print(f'  Open in browser: {profile_file}')

except Exception as e:
    print(f'⚠ Error: {str(e)[:200]}')
    import traceback
    print(traceback.format_exc()[:300])


DATA PROFILING REPORT

Generating data profile summary...
✓ Custom profiling report saved: reports/winequality-white-edited/data_profile_report.html
  File size: 0.01 MB
  Open in browser: reports/winequality-white-edited/data_profile_report.html


## 10. Generate Sweetviz Report

In [15]:
import os
import sweetviz as sv
import warnings

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("SWEETVIZ REPORT")
print("=" * 80)

try:
    print("\nGenerating Sweetviz report...")

    # Create report
    my_report = sv.analyze(df)

    # Output path
    sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report.html")

    # Save HTML report (correct method)
    my_report.show_html(
        filepath=sweetviz_file,
        open_browser=False,
        layout="widescreen",
        scale=1.0
    )

    print(f"✓ Sweetviz report saved: {sweetviz_file}")

    # File validation
    if os.path.exists(sweetviz_file):
        size_mb = os.path.getsize(sweetviz_file) / (1024 ** 2)
        print(f"  File size: {size_mb:.2f} MB")
    else:
        print("⚠ File was not created properly")

except ImportError:
    print("⚠ Sweetviz not installed.")
    print("Install using: pip install sweetviz")

except Exception as e:
    print(f"⚠ Error generating Sweetviz report:\n{e}")

    # Safe fallback (FIXED)
    try:
        print("\nTrying fallback method...")

        sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report_fallback.html")

        my_report.show_html(
            filepath=sweetviz_file,
            open_browser=False
        )

        print(f"✓ Fallback report saved: {sweetviz_file}")

    except Exception as e2:
        print(f"❌ Fallback also failed: {e2}")


SWEETVIZ REPORT

Generating Sweetviz report...


                                             |          | [  0%]   00:00 -> (? left)

Report reports/winequality-white-edited/sweetviz_report.html was generated.
✓ Sweetviz report saved: reports/winequality-white-edited/sweetviz_report.html
  File size: 1.28 MB


## 11. Generate AutoViz Visualizations

In [16]:
import os
from pathlib import Path
import warnings
from autoviz.AutoViz_Class import AutoViz_Class

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("AUTOVIZ VISUALIZATIONS")
print("=" * 80)

try:
    print("\nGenerating AutoViz plots...")

    autoviz_dir = os.path.join(dataset_output_dir, "autoviz_plots")
    Path(autoviz_dir).mkdir(parents=True, exist_ok=True)

    AV = AutoViz_Class()

    dft = AV.AutoViz(
        filename=DATA_FILE,
        sep=",",
        depVar="",
        dfte=None,
        header=0,
        verbose=1,
        lowess=False,
        chart_format="png",
        max_rows_analyzed=len(df),
        max_cols_analyzed=len(df.columns)
    )

    print("\n✓ AutoViz execution completed")

except ImportError:
    print("⚠ AutoViz is not installed.")

except Exception as e:
    print(f"⚠ Error during AutoViz execution:\n{e}")

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)

AUTOVIZ VISUALIZATIONS

Generating AutoViz plots...
Shape of your Data Set loaded: (3961, 12)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  11
    Number of Integer-Categorical Columns =  1
    Number of String-Categorical Columns =  0
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  0
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns = 

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
fixed_acidity,float64,0.000000,NA,3.800000,14.200000,Column has 106 outliers greater than upper bound (8.80) or lower than lower bound(4.80). Cap them or remove them.
volatile_acidity,float64,0.000000,NA,0.080000,1.100000,Column has 133 outliers greater than upper bound (0.51) or lower than lower bound(0.03). Cap them or remove them.
citric_acid,float64,0.000000,NA,0.000000,1.660000,Column has 223 outliers greater than upper bound (0.57) or lower than lower bound(0.09). Cap them or remove them.
residual_sugar,float64,0.000000,NA,0.600000,65.800000,Column has 16 outliers greater than upper bound (19.85) or lower than lower bound(-9.35). Cap them or remove them.
chlorides,float64,0.000000,NA,0.009000,0.346000,Column has 178 outliers greater than upper bound (0.07) or lower than lower bound(0.01). Cap them or remove them.
free_sulfur_dioxide,float64,0.000000,NA,2.000000,289.000000,Column has 44 outliers greater than upper bound (78.00) or lower than lower bound(-10.00). Cap them or remove them.
total_sulfur_dioxide,float64,0.000000,NA,9.000000,440.000000,Column has 14 outliers greater than upper bound (256.00) or lower than lower bound(16.00). Cap them or remove them.
density,float64,0.000000,NA,0.987110,1.038980,"Column has 6 outliers greater than upper bound (1.00) or lower than lower bound(0.99). Cap them or remove them., Column has a high correlation with ['residual_sugar']. Consider dropping one of them."
pH,float64,0.000000,NA,2.720000,3.820000,Column has 46 outliers greater than upper bound (3.59) or lower than lower bound(2.79). Cap them or remove them.
sulphates,float64,0.000000,NA,0.220000,1.080000,Column has 96 outliers greater than upper bound (0.76) or lower than lower bound(0.20). Cap them or remove them.


Number of All Scatter Plots = 66
All Plots done
Time to run AutoViz = 9 seconds 

 ###################### AUTO VISUALIZATION Completed ########################

✓ AutoViz execution completed


In [17]:
import matplotlib.pyplot as plt
import os

os.makedirs(autoviz_dir, exist_ok=True)

fig_nums = plt.get_fignums()

for i in fig_nums:
    fig = plt.figure(i)
    fig.savefig(f"{autoviz_dir}/autoviz_plot_{i}.png", bbox_inches="tight")

print(f"✓ Saved {len(fig_nums)} plots to: {autoviz_dir}")

✓ Saved 7 plots to: reports/winequality-white-edited/autoviz_plots


## 12. Launch D-Tale Interactive Dashboard

In [12]:

print('\n' + '='*80)
print('D-TALE INTERACTIVE DASHBOARD')
print('='*80)

try:
    import dtale
    import warnings
    warnings.filterwarnings('ignore')
    print('\nLaunching D-Tale dashboard...')
    
    d = dtale.show(df, open_browser=False)
    
    print(f'✓ D-Tale is running!')
    print("D-Tale URL:", d._main_url)
    print('\nD-Tale Features:')
    print('  - Interactive data exploration')
    print('  - Column statistics')
    print('  - Filtering & sorting')
    print('  - Visualization tools')
    print('\nNote: Keep this notebook running to access D-Tale.')
    print('\nTo stop D-Tale, uncomment and run the cleanup cell below.')
    
    dtale_instance = d

except ImportError:
    print('⚠ D-Tale not installed.')
    print('  Install: pip install dtale')
except Exception as e:
    print(f'⚠ Error launching D-Tale: {str(e)[:200]}')
    print('  Try updating: pip install --upgrade dtale')



D-TALE INTERACTIVE DASHBOARD

Launching D-Tale dashboard...
✓ D-Tale is running!
D-Tale URL: http://phoenix:40000/dtale/main/1

D-Tale Features:
  - Interactive data exploration
  - Column statistics
  - Filtering & sorting
  - Visualization tools

Note: Keep this notebook running to access D-Tale.

To stop D-Tale, uncomment and run the cleanup cell below.


## 13. Execution Summary

In [13]:
end_time = datetime.now()
execution_time = (end_time - start_time).total_seconds()

print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)

print(f'\n✓ EDA Pipeline finished successfully!')
print(f'  Execution Time: {execution_time:.2f} seconds')
print(f'  Output Directory: {dataset_output_dir}')

# List generated files
print(f'\nGenerated Reports:')
for item in os.listdir(dataset_output_dir):
    item_path = os.path.join(dataset_output_dir, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / 1024**2
        print(f'  ✓ {item} ({size_mb:.2f} MB)')
    elif os.path.isdir(item_path):
        file_count = len([f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))])
        print(f'  ✓ {item}/ ({file_count} files)')

print(f'\nOpen these reports in a web browser to explore the data!')


EXECUTION COMPLETE

✓ EDA Pipeline finished successfully!
  Execution Time: 17.54 seconds
  Output Directory: reports/winequality-white-edited

Generated Reports:
  ✓ autoviz_plots/ (0 files)
  ✓ data_profile_report.html (0.01 MB)
  ✓ sweetviz_report.html (0.00 MB)

Open these reports in a web browser to explore the data!


## 14. Optional Cleanup

In [14]:
# Uncomment to stop D-Tale
# if 'dtale_instance' in locals():
#     dtale_instance.kill()
#     print('D-Tale dashboard stopped.')